In [ ]:
# ENV: venv_rankllama
import pyterrier as pt

# Initialisation de PyTerrier avec suffisamment de RAM
if not pt.started():
    pt.init(mem=16000)

from pathlib import Path
from ir_datasets import load
from pyterrier.datasets import get_dataset

dataset = load("msmarco-passage/trec-dl-2019/judged")
pt_dataset = get_dataset("irds:msmarco-passage/trec-dl-2019/judged")

# %%
from typing import Any
from pyterrier.java import required
from pyterrier.terrier import IterDictIndexer

index_path = Path("./msmarco-index/")

@required
def index_document() -> Any:
    if not (index_path / "data.properties").exists():
        indexer = IterDictIndexer(str(index_path.absolute()))
        return indexer.index(pt_dataset.get_corpus_iter())  # type: ignore
    else:
        print("Already indexed.")
        from pyterrier import IndexRef
        return IndexRef.of(str(index_path.absolute()))  # type: ignore

index = index_document()  # type: ignore

# %%
from pyterrier.terrier import Retriever

bm25 = Retriever(
    index,
    wmodel="BM25",
    num_results=20,
    metadata=["docno", "text"],
)  # type: ignore

bm25.search("test").head()

# %%
from rankllama_reranker import RankLlamaReranker
from pyterrier.text import get_text

rankllama = RankLlamaReranker(verbose=True)

bm25_rankllama = bm25 % 10 >> get_text(pt_dataset, "text") >> rankllama
bm25_rankllama.search("test").head()

# %%
results_dir = Path("results/")
results_dir.mkdir(exist_ok=True, parents=True)

res_rankllama = bm25_rankllama.transform(pt_dataset.get_topics()[:50])

pt.io.write_results(
    res_rankllama,
    str(results_dir / "run_rankllama_test.trec"),
)

<h1 style="color: red; text-align: center; font-size: 2.5em;">
  🛑 STOP! CHANGE ENVIRONMENT NOW 🛑
</h1>

In [ ]:
# ENV: venv_ir_axioms
from pathlib import Path
import pyterrier as pt
from pyterrier.datasets import get_dataset, Dataset

if not pt.started():
    pt.init()

dataset_name = "msmarco-passage/trec-dl-2019/judged"
pt_dataset: Dataset = get_dataset(f"irds:{dataset_name}")

cache_dir = Path("cache/")
index_dir = cache_dir / "indices" / dataset_name.split("/")[0]
results_dir = Path("results/")

from pyterrier.terrier import Retriever
bm25 = Retriever(str(index_dir.absolute()), wmodel="BM25", metadata=["docno", "text"])

from ir_axioms.integrations.pyterrier.utils import inject_pyterrier
inject_pyterrier(index_location=index_dir, text_field="text", dataset=dataset_name)

res_rankllama = pt.io.read_results(str(results_dir / "run_rankllama_test.trec"))

from ir_axioms.axiom import (
    TFC1, LB1, PROX1, PROX2,
    ArgUC, QTArg, QTPArg, aSL, LNC1, TF_LNC, 
    PROX3, PROX4, PROX5, REG, ANTI_REG, ASPECT_REG, 
    AND, LEN_AND, M_AND, LEN_M_AND, DIV, LEN_DIV, 
    TFC3, M_TDC, LEN_M_TDC, STMC1, STMC2
)
from ir_axioms.integrations.pyterrier.experiment import AxiomaticExperiment

rankllama_pipeline = pt.apply.generic(
    lambda topics: res_rankllama[res_rankllama["qid"].isin(topics["qid"])].copy()
)

experiment = AxiomaticExperiment(
    retrieval_systems=[bm25, rankllama_pipeline],
    names=["BM25", "BM25 + RankLLaMA"],
    axioms = [
        TFC1().cached(cache_dir / "TFC1"), LB1().cached(cache_dir / "LB1"), 
        PROX1().cached(cache_dir / "PROX1"), PROX2().cached(cache_dir / "PROX2"),
        ArgUC().cached(cache_dir / "ArgUC"), QTArg().cached(cache_dir / "QTArg"), 
        QTPArg().cached(cache_dir / "QTPArg"), aSL().cached(cache_dir / "aSL"), 
        LNC1().cached(cache_dir / "LNC1"), TF_LNC().cached(cache_dir / "TF_LNC"),
        PROX3().cached(cache_dir / "PROX3"), PROX4().cached(cache_dir / "PROX4"), 
        PROX5().cached(cache_dir / "PROX5"), REG().cached(cache_dir / "REG"), 
        ANTI_REG().cached(cache_dir / "ANTI_REG"), ASPECT_REG().cached(cache_dir / "ASPECT_REG"),
        AND().cached(cache_dir / "AND"), LEN_AND().cached(cache_dir / "LEN_AND"), 
        M_AND().cached(cache_dir / "M_AND"), LEN_M_AND().cached(cache_dir / "LEN_M_AND"), 
        DIV().cached(cache_dir / "DIV"), LEN_DIV().cached(cache_dir / "LEN_DIV"),
        TFC3().cached(cache_dir / "TFC3"), M_TDC().cached(cache_dir / "M_TDC"),
        LEN_M_TDC().cached(cache_dir / "LEN_M_TDC"), STMC1().cached(cache_dir / "STMC1"), 
        STMC2().cached(cache_dir / "STMC2"),
    ],
    axiom_names = [
        "TFC1", "LB1", "PROX1", "PROX2",
        "ArgUC", "QTArg", "QTPArg", "aSL", "LNC1", "TF_LNC", "PROX3", "PROX4", "PROX5", 
        "REG", "ANTI_REG", "ASPECT_REG", "AND", "LEN_AND", "M_AND", "LEN_M_AND", "DIV", 
        "LEN_DIV", "TFC3", "M_TDC", "LEN_M_TDC", "STMC1", "STMC2",
    ],
    topics=pt_dataset.get_topics()[:50],
    qrels=pt_dataset.get_qrels(),
    depth=5,
    filter_by_qrels=True,
    text_field="text",
)

df = experiment.preferences
dist = experiment.preference_distribution.set_index("axiom")
consistency = experiment.preference_consistency.set_index("axiom").round(2)

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

N_TOPICS = 50
DEPTH = 5
TRUE_AXIOMS = [
    "TFC1", "LB1", "PROX1", "PROX2",
    "ArgUC", "QTArg", "QTPArg", "aSL", "LNC1", "TF_LNC", "PROX3", "PROX4", "PROX5", 
    "REG", "ANTI_REG", "ASPECT_REG", "AND", "LEN_AND", "M_AND", "LEN_M_AND", "DIV", 
    "LEN_DIV", "TFC3", "M_TDC", "LEN_M_TDC", "STMC1", "STMC2"
]
SYSTEMS = ["BM25", "BM25 + RankLLaMA"]

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#444444",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.6,
    "legend.fontsize": 10.5,
    "legend.frameon": False,
})

COLOR_BM25 = "#3C5A80"
COLOR_RANKLLAMA = "#B7472A"

def bar_labels(ax, fmt="{:.2f}", rotation=0, fontsize=10):
    for container in ax.containers:
        ax.bar_label(container, fmt=fmt, padding=3, fontsize=fontsize, rotation=rotation)

CHUNK_SIZE = 5
axiom_chunks = [TRUE_AXIOMS[i:i + CHUNK_SIZE] for i in range(0, len(TRUE_AXIOMS), CHUNK_SIZE)]

dist_full = experiment.preference_distribution.set_index("axiom")

for i, chunk in enumerate(axiom_chunks):
    dist_chunk = dist_full.loc[chunk]
    agreement_rate = dist_chunk["axiom == ORIG"] / dist_chunk.sum(axis=1)
    disagreement_rate = dist_chunk["axiom != ORIG"] / dist_chunk.sum(axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))
    x = range(len(chunk))
    
    ax.bar(x, agreement_rate, color=COLOR_BM25, label="BM25 + RankLLaMA en accord avec la règle", width=0.55)
    ax.bar(x, disagreement_rate, bottom=agreement_rate, color=COLOR_RANKLLAMA,
           label="BM25 + RankLLaMA en désaccord avec la règle", width=0.55)
    
    ax.set_xticks(list(x))
    ax.set_xticklabels(chunk, rotation=0, ha="center")
    ax.set_ylabel("Proportion des paires évaluées ")
    ax.set_title(f"Fréquence d'application des règles axiomatiques {i+1}")
    
    ax.set_ylim(0, 1)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig(results_dir / f"figure1_frequence_application_part{i+1}.png", dpi=250, bbox_inches="tight")
    plt.show()

consistency_full = experiment.preference_consistency.set_index("axiom").round(3)

for i, chunk in enumerate(axiom_chunks):
    consistency_plot = consistency_full.loc[chunk, ["BM25_consistency", "BM25 + RankLLaMA_consistency"]]
    consistency_plot.columns = SYSTEMS

    fig, ax = plt.subplots(figsize=(9, 5.5))
    consistency_plot.plot(kind="bar", ax=ax, color=[COLOR_BM25, COLOR_RANKLLAMA], width=0.7, edgecolor="white")
    
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Taux de conformité")
    ax.set_xlabel("Règles évaluées")
    ax.set_title(f"Taux de conformité des algorithmes aux règles {i+1}")
    
    plt.xticks(rotation=0, ha="center")
    bar_labels(ax, fmt="{:.2f}", rotation=0, fontsize=10)
    ax.legend(title=None, loc="upper right")
    plt.tight_layout()
    plt.savefig(results_dir / f"figure2_conformite_systeme_part{i+1}.png", dpi=250, bbox_inches="tight")
    plt.show()

rows = TRUE_AXIOMS + ["ORACLE"]
consistency_diff = (
    consistency_full.loc[rows, "BM25 + RankLLaMA_consistency"]
    - consistency_full.loc[rows, "BM25_consistency"]
).fillna(0).sort_values()

fig, ax = plt.subplots(figsize=(9, 12))
colors = [COLOR_RANKLLAMA if v >= 0 else COLOR_BM25 for v in consistency_diff]
bars = ax.barh(consistency_diff.index, consistency_diff.values, color=colors, edgecolor="white", height=0.7)

for container in ax.containers:
    ax.bar_label(container, fmt="%+.2f", padding=4, fontsize=9)
    
ax.axvline(0, color="#333333", linewidth=1)
xmax = max(abs(consistency_diff.min()), abs(consistency_diff.max())) * 1.35
ax.set_xlim(-xmax, xmax)

ax.set_xlabel("← BM25 plus conforme aux règles   |   RankLLaMA plus conforme aux règles →")
ax.set_title("Différence de conformité globale (RankLLaMA vs BM25)")
plt.tight_layout()
plt.savefig(results_dir / "figure3_difference_conformite.png", dpi=250, bbox_inches="tight")
plt.show()

inc_by_system = experiment.inconsistent_pairs.groupby("name").size().reindex(SYSTEMS)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
bars = ax.bar(SYSTEMS, inc_by_system.values, color=[COLOR_BM25, COLOR_RANKLLAMA], width=0.5, edgecolor="white")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=4, fontsize=12)
    
ax.set_ylabel("Nombre total de paires mal classées")
ax.set_ylim(0, inc_by_system.max() * 1.15)
ax.set_title("Erreurs d'ordonnancement par rapport aux jugements de pertinence (ORACLE)")
plt.tight_layout()
plt.savefig(results_dir / "figure4_volume_erreurs.png", dpi=250, bbox_inches="tight")
plt.show()